# RedditSIEM – Extremist Comment Classifier
### Demo su Google Colab

Pipeline: **entity detection → ABSA ensemble (DeBERTa + pyabsa) → detoxify → score**

> **Nota:** la prima esecuzione scarica i modelli (~2-4 GB). Usa **Runtime → Change runtime type → T4 GPU** per velocizzare l'inferenza.

## 1 · Installazione dipendenze

In [ ]:
%%capture
# Installa tutte le dipendenze
!pip install torch transformers detoxify nltk scipy
# pyabsa opzionale: se fallisce il classificatore usa solo DeBERTa
!pip install pyabsa || true

In [ ]:
import os, sys

REPO_URL = "https://github.com/5iraFic/RedditSIEM.git"
REPO_PATH = "/content/RedditSIEM"
BRANCH = "claude/extremist-comment-classifier-bQnxN"

# Clona se non esiste, altrimenti forza aggiornamento
if not os.path.exists(REPO_PATH):
    # Se il repo è privato: !git clone https://TUO_TOKEN@github.com/5iraFic/RedditSIEM.git /content/RedditSIEM
    !git clone {REPO_URL} {REPO_PATH}

os.chdir(REPO_PATH)
!git fetch origin {BRANCH} --quiet
!git checkout {BRANCH} --quiet
!git reset --hard origin/{BRANCH}   # forza l'allineamento all'ultimo commit

# Aggiunge il repo al path (idempotente)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Invalida la cache dei moduli Python per caricare il codice aggiornato
for mod in list(sys.modules.keys()):
    if mod.startswith("extremist_classifier"):
        del sys.modules[mod]

print("Ultimo commit:")
!git log --oneline -3

## 2 · Caricamento del classificatore

I modelli vengono scaricati automaticamente alla prima chiamata:
- `yangheng/deberta-v3-base-absa-v1.1` (~600 MB)
- `pyabsa` ATEPC multilingual checkpoint (~300 MB)
- `detoxify` multilingual XLM-RoBERTa (~1.1 GB)

In [ ]:
import logging

# Mostra solo WARNING e sopra (nasconde i LOAD REPORT verbosi dei modelli)
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
# Silenzia completamente i warning ripetuti di pyabsa
logging.getLogger("extremist_classifier.absa").setLevel(logging.ERROR)

from extremist_classifier import ExtremistCommentClassifier

clf = ExtremistCommentClassifier(
    use_deberta     = True,
    use_pyabsa      = True,     # tenta pyabsa; se non disponibile usa solo DeBERTa
    deberta_weight  = 0.6,
    pyabsa_weight   = 0.4,
    absa_weight     = 0.6,
    toxicity_weight = 0.4,
    detoxify_model  = "multilingual",
    device          = "auto",
)
print("✓ Classificatore pronto")

## 3 · Classifica un singolo commento

In [ ]:
comment = (
    "We must support the mujahideen in their jihad against the kuffar. "
    "May Allah grant them victory and destroy the infidels!"
)

result = clf.classify(
    comment    = comment,
    comment_id = "t1_demo001",
    subreddit  = "r/example",
    author     = "u/testuser",
)

print(result.summary())

In [ ]:
import json
# Output strutturato pronto per SIEM (Elasticsearch / Splunk)
print(json.dumps(result.to_siem_event(), indent=2))

## 4 · Batch evaluation su commenti di test

In [ ]:
SAMPLES = [
    # (testo, rischio_atteso)
    # ✅ Glorificazione diretta + entità multiple → HIGH
    ("We must support the mujahideen in their jihad against the kuffar.",              "HIGH"),
    ("ISIS is right to establish the caliphate. Death to Jews and infidels.",           "HIGH"),
    # ⚠️  Una sola entità con glorificazione lessicale ma frase ambigua
    #     (potrebbe essere una citazione giornalistica) → MEDIUM è corretto
    ("The shaheed who carried out the operation is a true martyr.",                    "MEDIUM"),
    # ✅ Menzione critica/contestuale → LOW
    ("People say ISIS is brutal, but remember what Americans did in Iraq.",             "LOW"),
    # ✅ Counter-narrative esplicita → NONE
    ("ISIS must be stopped. Their attacks on civilians are unacceptable.",              "NONE"),
    ("I visited the mosque last Friday. The sermon was about patience.",                "NONE"),
    ("Just made some amazing shawarma for dinner. Highly recommend!",                  "NONE"),
]

batch = [{"body": t, "id": f"t1_{i:03d}"} for i, (t, _) in enumerate(SAMPLES)]
results = clf.classify_batch(batch)

print(f"{'✓/✗':<4} {'Label':>8}  {'Expected':>8}  {'Score':>6}  Entità trovate")
print("-" * 65)
correct = 0
for (text, expected), res in zip(SAMPLES, results):
    ok = "✓" if res.risk_label == expected else "✗"
    if res.risk_label == expected:
        correct += 1
    entities = ", ".join(res.entity_labels_found) or "—"
    print(f"{ok:<4} {res.risk_label:>8}  {expected:>8}  {res.doc_score:>6.3f}  {entities}")

print(f"\n{correct}/{len(SAMPLES)} corretti")

## 5 · Classifica un tuo commento personalizzato

In [ ]:
# ← Modifica questo testo con il commento che vuoi testare
MY_COMMENT = "Insert your Reddit comment here..."

r = clf.classify(MY_COMMENT)
print(r.summary())
print()

# Dettaglio per ogni entità trovata
for s in r.sentences:
    print(f"  Entità : {s.entity_label!r} ({s.entity_type})")
    print(f"  Frase  : {s.sentence!r}")
    print(f"  ABSA   : {s.absa_sentiment} (conf={s.absa_confidence:.3f}, agreement={s.absa_agreement})")
    print(f"  Tox    : toxicity={s.toxicity_scores.get('toxicity',0):.3f}  "
          f"identity_attack={s.toxicity_scores.get('identity_attack',0):.3f}  "
          f"threat={s.toxicity_scores.get('threat',0):.3f}")
    print(f"  Signal : {s.signal_type}  score={s.sentence_extremism_score:.3f}")
    print()

## 6 · (Opzionale) Solo DeBERTa, senza pyabsa

Se pyabsa è lento o dà problemi, puoi disabilitarlo e usare solo DeBERTa:

## 7 · Demo: estrazione frase/sotto-frase da testo lungo

## 8 · Analisi del dataset ISIS_Seed_Complete.csv

In [ ]:
# Carica il file CSV da locale
from google.colab import files
import pandas as pd
import io

print("Seleziona ISIS_Seed_Complete.csv dal tuo computer...")
uploaded = files.upload()

filename = next(iter(uploaded))
raw = uploaded[filename]

# Auto-detect separatore (prova tab poi virgola)
for sep in ("\t", ",", ";"):
    df = pd.read_csv(io.BytesIO(raw), sep=sep, encoding="utf-8")
    if len(df.columns) > 2:
        break

# Rimuovi eventuali colonne vuote create da separatori finali
df = df.loc[:, df.columns.str.strip() != ""]
df.columns = df.columns.str.strip()

print(f"\n✓ Dataset caricato: {len(df)} righe  |  sep={repr(sep)}")
print(f"Colonne: {list(df.columns)}")
print(f"\nLabel distribution:\n{df['Label'].value_counts().to_string()}")
df.head(3)

In [ ]:
import time

# ── Parametri ───────────────────────────────────────────────────────────────
MAX_ROWS   = None   # None = tutto il dataset; metti es. 100 per un test rapido
BATCH_SIZE = 32     # righe per batch (bilancia velocità e memoria)
# ─────────────────────────────────────────────────────────────────────────────

subset = df.copy() if MAX_ROWS is None else df.head(MAX_ROWS).copy()
texts  = subset["Text"].fillna("").tolist()
labels = subset["Label"].tolist()

print(f"Classificazione di {len(texts)} testi (batch_size={BATCH_SIZE})...")
t0 = time.time()

all_results = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = [{"body": t, "id": str(i + j)} for j, t in enumerate(texts[i:i+BATCH_SIZE])]
    all_results.extend(clf.classify_batch(batch))
    done = min(i + BATCH_SIZE, len(texts))
    elapsed = time.time() - t0
    speed = done / elapsed
    eta = (len(texts) - done) / speed if speed > 0 else 0
    print(f"  {done}/{len(texts)}  ({speed:.1f} testi/s  ETA {eta:.0f}s)", end="\r")

print(f"\n✓ Completato in {time.time()-t0:.1f}s")

In [ ]:
long_comment = """
I was just reading about the history of the region.
The weather here has been really nice lately, quite sunny.
We must support the mujahideen in their holy struggle; they are true warriors of faith.
Yesterday I had pasta for dinner and it was delicious.
Death to all kuffar and infidels who oppose the caliphate!
The local football team won again, great result.
"""

r = clf.classify(long_comment)

print(f"Testo: {len(long_comment.split())} parole, {len(long_comment.splitlines())} righe")
print(f"Frasi analizzate con entità: {len(r.sentences)}")
print()

for s in r.sentences:
    print(f"  ── Entità rilevata: [{s.entity_type.upper()}] {s.entity_label!r}")
    print(f"     Sotto-frase passata a ABSA+detoxify:")
    print(f"     → {s.sentence!r}")
    print(f"     Sentiment: {s.absa_sentiment} (conf={s.absa_confidence:.2f})  "
          f"Signal: {s.signal_type}  Score: {s.sentence_extremism_score:.3f}")
    print()

print(result.summary() if False else r.summary())

In [ ]:
import pandas as pd

# Aggiungi le colonne di classificazione al dataframe
subset["clf_risk"]     = [r.risk_label   for r in all_results]
subset["clf_score"]    = [r.doc_score    for r in all_results]
subset["clf_entities"] = [", ".join(r.entity_labels_found) for r in all_results]
subset["clf_flagged"]  = [" | ".join(r.flagged_sentences)  for r in all_results]

# ── Statistiche generali ────────────────────────────────────────────────────
print("=" * 55)
print("Distribuzione risk_label sul dataset")
print("=" * 55)
risk_counts = subset["clf_risk"].value_counts()
total = len(subset)
for label, cnt in risk_counts.items():
    bar = "█" * int(cnt / total * 40)
    print(f"  {label:8s} {cnt:5d}  ({cnt/total*100:5.1f}%)  {bar}")

print()

# ── Breakdown per Label del dataset ────────────────────────────────────────
print("=" * 55)
print("Risk per Label originale del dataset")
print("=" * 55)
pivot = pd.crosstab(subset["Label"], subset["clf_risk"])
# Ordina colonne per gravità
col_order = [c for c in ["HIGH", "MEDIUM", "LOW", "NONE"] if c in pivot.columns]
print(pivot[col_order].to_string())

print()

# ── Testi HIGH risk ────────────────────────────────────────────────────────
high = subset[subset["clf_risk"] == "HIGH"].sort_values("clf_score", ascending=False)
print(f"{'='*55}")
print(f"Testi HIGH risk: {len(high)} ({len(high)/total*100:.1f}%)")
print(f"{'='*55}")
for _, row in high.head(10).iterrows():
    print(f"\n  Score: {row['clf_score']:.3f}  |  Entità: {row['clf_entities']}")
    print(f"  Testo: {str(row['Text'])[:120]!r}")
    if row['clf_flagged']:
        print(f"  Flag : {row['clf_flagged'][:100]!r}")

In [ ]:
# Esporta il dataframe arricchito con i punteggi del classificatore
out_path = "/content/ISIS_Seed_classified.csv"
subset.to_csv(out_path, index=False, sep="\t")
print(f"✓ Salvato: {out_path}")

# Scarica il file
files.download(out_path)

In [ ]:
clf_lite = ExtremistCommentClassifier(
    use_deberta    = True,
    use_pyabsa     = False,   # solo DeBERTa
    detoxify_model = "multilingual",
)

r = clf_lite.classify("The mujahideen are heroes fighting the kuffar.")
print(r.summary())